# 混合数据集

![](./xiaodongguaAIGC_dataset.png)

In [2]:
from datasets import load_dataset, concatenate_datasets, DatasetDict

data_name1 = 'xiaodongguaAIGC/alpaca_gpt4_data_zh' # ignore
data_name2 = 'vicgalle/alpaca-gpt4'
data_name3 = 'LooksJuicy/ruozhiba'
data_name4 = 'silk-road/alpaca-data-gpt4-chinese' # 基于google翻译 仅演示如何按照比例来切割数据集

dataset1 = load_dataset(data_name1)
dataset2 = load_dataset(data_name2)
dataset3 = load_dataset(data_name3)
dataset4 = load_dataset(data_name4)

# dataset1: 中文alpaca数据集

In [4]:
dataset1 = load_dataset(data_name1)
print(dataset1)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 48818
    })
})


# dataset2: 英文 alpaca 数据集

In [5]:
# process dataset2
print(dataset2)
dataset2 = dataset2.remove_columns([
    'text',
])
print(dataset2)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 52002
    })
})
DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 52002
    })
})


# dataset3:  弱智吧QA

In [6]:
# process dataset3
print(dataset3)
def process_ruozhiba(examples):
    examples['input'] = ''
    return examples
dataset3 = dataset3.map(process_ruozhiba, num_proc=8)
print(dataset3)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output'],
        num_rows: 1496
    })
})


Map (num_proc=8):   0%|          | 0/1496 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'input'],
        num_rows: 1496
    })
})


# dataset4:  另一个alpaca中文数据

数据质量差不加入

In [7]:
print(dataset4)
def process_fn(examples):
    examples['instruction']=examples['instruction_zh']
    examples['input']=examples['input_zh']
    examples['output']=examples['output_zh']
    return examples
dataset4 = dataset4.map(process_fn, num_proc=8, remove_columns = ["instruction_zh", "input_zh", 'output_zh'])
dataset4['train'] = dataset4['train'].shard(num_shards=10, index=0) # 只取10%数据
print(dataset4)

DatasetDict({
    train: Dataset({
        features: ['instruction_zh', 'input_zh', 'output_zh', 'instruction', 'input', 'output'],
        num_rows: 52049
    })
})


Map (num_proc=8):   0%|          | 0/52049 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 5205
    })
})


In [8]:
# 规则过滤: 基于长度过滤
print(dataset4)
dataset4['train'] = dataset4['train'].filter(lambda example: len(example["instruction"]+
                                                                 example["input"]+
                                                                 example["output"])<512)
print(dataset4)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 5205
    })
})


Filter:   0%|          | 0/5205 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 4670
    })
})


# dataset5. 英文选择题 allenai/ai2_arc

In [9]:
data_name5 = 'allenai/ai2_arc'
dataset5 = load_dataset(data_name5, 'ARC-Challenge')
dataset6 = load_dataset(data_name5, 'ARC-Easy')
print(dataset5)
print(dataset6)

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/2251 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2376 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 1119
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 1172
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 299
    })
})
DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 2251
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 2376
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey'],
        num_rows: 570
    })
})


In [10]:
print(dataset5['train'][0])

{'id': 'Mercury_SC_415702', 'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?', 'choices': {'text': ['dry palms', 'wet palms', 'palms covered with oil', 'palms covered with lotion'], 'label': ['A', 'B', 'C', 'D']}, 'answerKey': 'A'}


In [11]:
mcq_prompt = 'There is a single choice question. Answer the question by replying A, B, C or D.\n'
def process_arc(examples):

    prompt_q = 'Question: ' + examples['question']
    for label, text  in zip(examples['choices']['label'], examples['choices']['text']):
               prompt_q = prompt_q + label + '. ' + text + '\n'
    prompt_a = examples['answerKey']
    examples['instruction'] = mcq_prompt
    examples['input'] = prompt_q
    examples['output'] = prompt_a
    return examples


dataset5_alpaca = dataset5.map(process_arc)
print(dataset5_alpaca)
dataset6_alpaca = dataset6.map(process_arc)
print(dataset6_alpaca)


Map:   0%|          | 0/1119 [00:00<?, ? examples/s]

Map:   0%|          | 0/1172 [00:00<?, ? examples/s]

Map:   0%|          | 0/299 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 1119
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 1172
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 299
    })
})


Map:   0%|          | 0/2251 [00:00<?, ? examples/s]

Map:   0%|          | 0/2376 [00:00<?, ? examples/s]

Map:   0%|          | 0/570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 2251
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 2376
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 570
    })
})


In [12]:
print(dataset5_alpaca['train'][0])

{'id': 'Mercury_SC_415702', 'question': 'George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?', 'choices': {'text': ['dry palms', 'wet palms', 'palms covered with oil', 'palms covered with lotion'], 'label': ['A', 'B', 'C', 'D']}, 'answerKey': 'A', 'instruction': 'There is a single choice question. Answer the question by replying A, B, C or D.\n', 'input': 'Question: George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?A. dry palms\nB. wet palms\nC. palms covered with oil\nD. palms covered with lotion\n', 'output': 'A'}


In [13]:
dataset5_alpaca['train'] = dataset5_alpaca['train'].filter(lambda example: len(example["choices"]['label'])==4 )
dataset5_alpaca['test'] = dataset5_alpaca['test'].filter(lambda example: len(example["choices"]['label'])==4 )
dataset5_alpaca['validation'] = dataset5_alpaca['validation'].filter(lambda example: len(example["choices"]['label'])==4 )
dataset6_alpaca['train'] = dataset6_alpaca['train'].filter(lambda example: len(example["choices"]['label'])==4 )
dataset6_alpaca['test'] = dataset6_alpaca['test'].filter(lambda example: len(example["choices"]['label'])==4 )
dataset6_alpaca['validation'] = dataset6_alpaca['validation'].filter(lambda example: len(example["choices"]['label'])==4 )
print(dataset5_alpaca)

Filter:   0%|          | 0/1119 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1172 [00:00<?, ? examples/s]

Filter:   0%|          | 0/299 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2251 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2376 [00:00<?, ? examples/s]

Filter:   0%|          | 0/570 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 1117
    })
    test: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 1165
    })
    validation: Dataset({
        features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output'],
        num_rows: 295
    })
})


In [14]:
dataset6_alpaca = dataset6_alpaca.remove_columns(["id", "question", 'choices', 'answerKey'])
dataset5_alpaca = dataset5_alpaca.remove_columns(["id", "question", 'choices', 'answerKey'])
print(dataset5_alpaca)
print(dataset6_alpaca)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1117
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1165
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 295
    })
})
DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2241
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2365
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 567
    })
})


# 7. 中文选择题+COT

In [15]:
# ZhangRC/chinese-multi-choice-ceval-validation-glm4-explanation
data_name7 = 'ZhangRC/chinese-multi-choice-ceval-validation-glm4-explanation'
dataset7 = load_dataset(data_name7)
print(dataset7)
print(dataset7['train'][0])

Generating train split:   0%|          | 0/1176 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'A', 'B', 'C', 'D', 'answer', 'explanation', 'domain'],
        num_rows: 1176
    })
})
{'id': 0, 'question': '下列关于税法基本原则的表述中，不正确的是____。', 'A': '税收法定原则包括税收要件法定原则和税务合法性原则', 'B': '税收公平原则源于法律上的平等性原则', 'C': '税收效率原则包含经济效率和行政效率两个方面', 'D': '税务机关按法定程序依法征税，可以自由做出减征、停征或免征税款的决定', 'answer': 'D', 'explanation': '**分析和解读：**\n\nA. **税收法定原则：** 这是税法的一个基本原则，指的是税收征收必须有法律依据，包括税收的要件（如税率、税基、税收对象等）必须由法律规定，以及税务行为必须符合法律程序。这个选项正确地表述了税收法定原则的两个方面：税收要件法定原则和税务合法性原则。\n\nB. **税收公平原则：** 这个原则强调的是税收负担应根据纳税人的经济能力分配，即能力相同的人应承担相同的税负，能力不同的人应承担不同的税负。它确实源于法律上的平等性原则，即相似的情况应得到相似的处理。\n\nC. **税收效率原则：** 这个原则有两个方面：经济效率和行政效率。经济效率关注税收对经济活动的影响，应尽量减少对经济活动的扭曲；行政效率关注税收管理的成本，应尽量降低税收征管的成本和提高税收征管效率。这个选项正确地描述了税收效率原则的两个方面。\n\nD. **税务机关的权力：** 税务机关在征税过程中必须遵守法定程序，不能随意减征、停征或免征税款。任何税收减免都必须有明确的法律依据，税务机关没有自由做出这些决定的权力。\n\n**权威正确答案：**\n\nD. 税务机关按法定程序依法征税，可以自由做出减征、停征或免征税款的决定\n\n这个选项的表述是不正确的。根据税收法定原则，税务机关必须依据法律的规定进行征税，不能自由做出减征、停征或免征税款的决定，这样的决定需要有明确的法律授权或依据。因此，正确答案是D。', 'domain': 'a

In [16]:
mcq_prompt = '以下是单项选择题，请直接给出正确答案的选项A,B,C,D。\n'
mc_reason_prompt = '以下是单项选择题，请详细解答,在最后给出正确答案的选项A,B,C,D。\n'
def process_cmmlu(examples):
    prompt_q = '题目:' + examples['question'] + '\n' \
                + 'A. ' + examples['A'] + '\n' \
                + 'B. ' + examples['B'] + '\n' \
                + 'C. ' + examples['C'] + '\n' \
                + 'D. ' + examples['D'] + '\n'
    # prompt_a = examples['answer']
    if examples['id']%10 == 0:
        examples['instruction'] = mc_reason_prompt
        prompt_a = examples['explanation']
    else:
        examples['instruction'] = mcq_prompt
        prompt_a = examples['answer']
    # examples['instruction'] = mcq_prompt
    examples['input'] = prompt_q
    examples['output'] = prompt_a
    return examples

dataset7_alpaca = dataset7.map(process_cmmlu, remove_columns=['id','question','A','B','C','D','answer','explanation','domain'])
print(dataset7_alpaca)

Map:   0%|          | 0/1176 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1176
    })
})


In [17]:
print(dataset7_alpaca['train'][0]) # CoT
print(dataset7_alpaca['train'][1])

{'instruction': '以下是单项选择题，请详细解答,在最后给出正确答案的选项A,B,C,D。\n', 'input': '题目:下列关于税法基本原则的表述中，不正确的是____。\nA. 税收法定原则包括税收要件法定原则和税务合法性原则\nB. 税收公平原则源于法律上的平等性原则\nC. 税收效率原则包含经济效率和行政效率两个方面\nD. 税务机关按法定程序依法征税，可以自由做出减征、停征或免征税款的决定\n', 'output': '**分析和解读：**\n\nA. **税收法定原则：** 这是税法的一个基本原则，指的是税收征收必须有法律依据，包括税收的要件（如税率、税基、税收对象等）必须由法律规定，以及税务行为必须符合法律程序。这个选项正确地表述了税收法定原则的两个方面：税收要件法定原则和税务合法性原则。\n\nB. **税收公平原则：** 这个原则强调的是税收负担应根据纳税人的经济能力分配，即能力相同的人应承担相同的税负，能力不同的人应承担不同的税负。它确实源于法律上的平等性原则，即相似的情况应得到相似的处理。\n\nC. **税收效率原则：** 这个原则有两个方面：经济效率和行政效率。经济效率关注税收对经济活动的影响，应尽量减少对经济活动的扭曲；行政效率关注税收管理的成本，应尽量降低税收征管的成本和提高税收征管效率。这个选项正确地描述了税收效率原则的两个方面。\n\nD. **税务机关的权力：** 税务机关在征税过程中必须遵守法定程序，不能随意减征、停征或免征税款。任何税收减免都必须有明确的法律依据，税务机关没有自由做出这些决定的权力。\n\n**权威正确答案：**\n\nD. 税务机关按法定程序依法征税，可以自由做出减征、停征或免征税款的决定\n\n这个选项的表述是不正确的。根据税收法定原则，税务机关必须依据法律的规定进行征税，不能自由做出减征、停征或免征税款的决定，这样的决定需要有明确的法律授权或依据。因此，正确答案是D。'}
{'instruction': '以下是单项选择题，请直接给出正确答案的选项A,B,C,D。\n', 'input': '题目:甲公司是国内一家领先的新媒体、通信及移动增值服务公司，由于遭受世界金融危机，甲公司经济利润严重下滑，经营面临困境，但为了稳定职工队伍，公司并未进行裁员，而是实行高层管理人员减薪措施。甲公司此举

# 8. code

In [19]:
dataset_name_code = 'flytech/python-codes-25k' # 50k
dataset_code = load_dataset(dataset_name_code)
print(dataset_code)

def process_fn_code(examples):
    examples['instruction']='Write a python code to response question. '
    return examples
dataset_code = dataset_code.map(process_fn_code, num_proc=24)

# todo: 增加instruction描述
# 该数据对sft影响大，
dataset_code['train'] = dataset_code['train'].shard(num_shards=100, index=0) # 只取20%数据 10k
dataset_code['train'] = dataset_code['train'].remove_columns(['text'])

DatasetDict({
    train: Dataset({
        features: ['text', 'input', 'instruction', 'output'],
        num_rows: 49626
    })
})


In [20]:
print(dataset_code)

DatasetDict({
    train: Dataset({
        features: ['input', 'instruction', 'output'],
        num_rows: 9926
    })
})


# 9. meta-math

In [25]:
dataset_name_math = 'meta-math/MetaMathQA' # 395k
dataset_math = load_dataset(dataset_name_math)
print(dataset_math)

def process_math_fn(examples):
    examples['instruction']=examples['query']
    examples['input']=''
    examples['output']=examples['response']
    return examples
dataset_math = dataset_math.map(process_math_fn, num_proc=8, remove_columns = ["query", "response", 'original_question', 'type'])

dataset_math['train'] = dataset_math['train'].shard(num_shards=40, index=0)
# dataset_math['train'] = dataset_math['train'].remove_column(['text'])

DatasetDict({
    train: Dataset({
        features: ['type', 'query', 'original_question', 'response'],
        num_rows: 395000
    })
})


In [26]:
print(dataset_math)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 9875
    })
})


# 合并数据集

In [27]:
dataset = concatenate_datasets([dataset1['train'], dataset2['train'], dataset3['train'],
                               dataset5_alpaca['train'], dataset5_alpaca['validation'], dataset5_alpaca['test'],
                               dataset6_alpaca['train'], dataset6_alpaca['validation'], dataset6_alpaca['test'],
                               dataset7_alpaca['train'],
                               dataset_code['train'],
                               dataset_math['train']])
dataset = DatasetDict({'train': dataset})
dataset = dataset.shuffle(seed=42)
print(dataset)
# print(dataset['train'][:20])
for i in range(20):
    print('-'*100)
    print('instruction: ', dataset['train']['instruction'][i])
    print('input: ', dataset['train']['input'][i])
    print('output: ', dataset['train']['output'][i])

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 131043
    })
})
----------------------------------------------------------------------------------------------------
instruction:  Summarize the main reasons why small businesses need a website.
input:  
output:  A small business needs a website to establish an online presence, attract a wider audience and reach customers beyond their physical location, increase credibility and trust in their brand, provide a platform for easy access to information about their products and services, improve their customer experience, as well as to engage and interact with customers using real-time communication methods. Overall, a well-designed website can help small businesses stay competitive within their industries and secure their long-term success.
----------------------------------------------------------------------------------------------------
instruction:  Generate five guiding principl

# 保存

In [28]:
dataset.save_to_disk('./output/awesome-sft')

Saving the dataset (0/1 shards):   0%|          | 0/131043 [00:00<?, ? examples/s]

In [ ]:
# get token from https://huggingface.co/settings/tokens
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 

In [34]:
from datasets import load_from_disk
my_data = './output/awesome-sft'
mix_dataset = load_from_disk(my_data)

In [35]:
mix_dataset.push_to_hub("YOU_HF_NAME/awesome-sft")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/132 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/xiaodongguaAIGC/awesome-sft/commit/06d6832de1ba9fb90bfc68a0ed4ad27f700fc648', commit_message='Upload dataset', commit_description='', oid='06d6832de1ba9fb90bfc68a0ed4ad27f700fc648', pr_url=None, pr_revision=None, pr_num=None)